In [6]:
import pandas as pd
from bug_severity_classifier.rules import guess_severity

In [7]:
bugs = pd.DataFrame({
    "title": [
        "App crash on login",
        "Login button broken",
        "Typo in footer",
        "Data loss on save",
        "Search timeout",
    ],
})
bugs

,title
0,App crash on login
1,Login button broken
2,Typo in footer
3,Data loss on save
4,Search timeout


In [8]:
bugs["severity"] = bugs["title"].apply(guess_severity)
bugs

,title,severity
0,App crash on login,critical
1,Login button broken,major
2,Typo in footer,minor
3,Data loss on save,critical
4,Search timeout,major


In [9]:
bugs["severity"].value_counts()

severity
critical    2
major       2
minor       1
Name: count, dtype: int64

In [10]:
df = pd.read_csv("../data/bugs_sample.csv")
df

,title,severity
0,App crash on login,critical
1,Data loss when saving draft,critical
2,Users can see other users' invoices,critical
3,Payment fails silently,critical
4,Login button broken,major
5,Search timeout after 30 seconds,major
6,Export to PDF fails,major
7,Filters reset when changing page,major
8,Typo in footer,minor
9,Error message has a typo,minor


In [11]:
df["predicted"] = df["title"].apply(guess_severity)
df

,title,severity,predicted
0,App crash on login,critical,critical
1,Data loss when saving draft,critical,critical
2,Users can see other users' invoices,critical,minor
3,Payment fails silently,critical,major
4,Login button broken,major,major
5,Search timeout after 30 seconds,major,major
6,Export to PDF fails,major,major
7,Filters reset when changing page,major,minor
8,Typo in footer,minor,minor
9,Error message has a typo,minor,major


In [14]:
df.groupby("severity")["correct"].mean()

severity
critical    0.50
major       0.75
minor       0.75
Name: correct, dtype: float64

In [15]:
eclipse = pd.read_csv("../data/bugs_eclipse.csv")
eclipse.shape

(88682, 5)

In [16]:
eclipse.head()

,Project,Bug ID,Severity Label,Resolution Status,Short Description
0,Bugzilla,322082,normal,FIXED,Clean up user selection SQL
1,Bugzilla,322412,trivial,FIXED,Typo in error message
2,Bugzilla,322315,normal,FIXED,If logged out $user->can_bless gives: Not an A...
3,Bugzilla,322244,trivial,FIXED,cookies are incorrectly detainted when logging...
4,Bugzilla,322212,critical,FIXED,DERIVE_GROUPS_TABLES_ALREADY_LOCKED is ignored...


In [17]:
df["correct"] = df["predicted"] == df["severity"]
df

,title,severity,predicted,correct
0,App crash on login,critical,critical,True
1,Data loss when saving draft,critical,critical,True
2,Users can see other users' invoices,critical,minor,False
3,Payment fails silently,critical,major,False
4,Login button broken,major,major,True
5,Search timeout after 30 seconds,major,major,True
6,Export to PDF fails,major,major,True
7,Filters reset when changing page,major,minor,False
8,Typo in footer,minor,minor,True
9,Error message has a typo,minor,major,False


In [18]:
pd.crosstab(df["severity"], df["predicted"])

predicted,critical,major,minor
severity,,,
critical,2,1,1
major,0,3,1
minor,0,1,3


In [19]:
eclipse = pd.read_csv("../data/bugs_eclipse.csv")
eclipse.shape

(88682, 5)

In [20]:
eclipse.head()

,Project,Bug ID,Severity Label,Resolution Status,Short Description
0,Bugzilla,322082,normal,FIXED,Clean up user selection SQL
1,Bugzilla,322412,trivial,FIXED,Typo in error message
2,Bugzilla,322315,normal,FIXED,If logged out $user->can_bless gives: Not an A...
3,Bugzilla,322244,trivial,FIXED,cookies are incorrectly detainted when logging...
4,Bugzilla,322212,critical,FIXED,DERIVE_GROUPS_TABLES_ALREADY_LOCKED is ignored...


In [21]:
eclipse["Severity Label"].value_counts()

Severity Label
normal      72170
critical     5936
major        4573
minor        3125
trivial      2080
blocker       798
Name: count, dtype: int64

In [22]:
(eclipse["Severity Label"] == "normal").mean()

np.float64(0.8138066349428295)

In [23]:
mapping = {
    "blocker": "critical",
    "critical": "critical",
    "major": "major",
    "normal": "major",
    "minor": "minor",
    "trivial": "minor",
}
eclipse["severity"] = eclipse["Severity Label"].map(mapping)
eclipse["severity"].value_counts()

severity
major       76743
critical     6734
minor        5205
Name: count, dtype: int64

In [24]:
eclipse["predicted"] = eclipse["Short Description"].fillna("").apply(guess_severity)
eclipse["correct"] = eclipse["predicted"] == eclipse["severity"]
eclipse["correct"].mean()

np.float64(0.17862700435263074)

In [25]:
eclipse.groupby("severity")["correct"].mean()

severity
critical    0.474606
major       0.102368
minor       0.920077
Name: correct, dtype: float64

In [26]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    eclipse, test_size=0.2, random_state=42, stratify=eclipse["severity"]
)
len(train), len(test)

(70945, 17737)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
model.fit(train["Short Description"].fillna(""), train["severity"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['critical','major','minor']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [29]:
predictions = model.predict(test["Short Description"].fillna(""))
(predictions == test["severity"]).mean()

np.float64(0.894288774877375)

In [30]:
import numpy as np

words = model[0].get_feature_names_out()
classifier = model[1]

for i, label in enumerate(classifier.classes_):
    best = np.argsort(classifier.coef_[i])[-10:][::-1]
    print(label, ":", [words[j] for j in best])

critical : ['crash', 'crashes', 'failure', 'assertion', 'hang', 'abort', 'bugzilla', 'crashing', 'hangs', 'differential']
major : ['ionmonkey', 'gcli', 'mingw', 'investigate', 'implement', 'android', '0010', 'intermittent', 'showing', 'add']
minor : ['javadoc', 'typo', 'remove', 'should', 'useless', 'dvt33', 'mnemonic', 'minor', 'slightly', 'warning']


In [31]:
import numpy as np

words = model[0].get_feature_names_out()
classifier = model[1]

for i, label in enumerate(classifier.classes_):
    best = np.argsort(classifier.coef_[i])[-10:][::-1]
    print(label, ":", [words[j] for j in best])

critical : ['crash', 'crashes', 'failure', 'assertion', 'hang', 'abort', 'bugzilla', 'crashing', 'hangs', 'differential']
major : ['ionmonkey', 'gcli', 'mingw', 'investigate', 'implement', 'android', '0010', 'intermittent', 'showing', 'add']
minor : ['javadoc', 'typo', 'remove', 'should', 'useless', 'dvt33', 'mnemonic', 'minor', 'slightly', 'warning']


In [32]:
predictions = model.predict(test["Short Description"].fillna(""))
(predictions == test["severity"]).mean()

np.float64(0.894288774877375)

In [33]:
(predictions == test["severity"]).groupby(test["severity"]).mean()

severity
critical    0.558278
major       0.981953
minor       0.036503
Name: severity, dtype: float64

In [34]:
model_balanced = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
model_balanced.fit(train["Short Description"].fillna(""), train["severity"])

predictions_balanced = model_balanced.predict(test["Short Description"].fillna(""))
print("Score global :", (predictions_balanced == test["severity"]).mean())
(predictions_balanced == test["severity"]).groupby(test["severity"]).mean()

Score global : 0.7274059874837909


severity
critical    0.753526
major       0.740895
minor       0.494717
Name: severity, dtype: float64

In [35]:
from sklearn.metrics import classification_report

print(classification_report(test["severity"], predictions_balanced))

              precision    recall  f1-score   support

    critical       0.45      0.75      0.57      1347
       major       0.94      0.74      0.83     15349
       minor       0.15      0.49      0.23      1041

    accuracy                           0.73     17737
   macro avg       0.51      0.66      0.54     17737
weighted avg       0.85      0.73      0.77     17737



In [36]:
print(classification_report(test["severity"], predictions))

              precision    recall  f1-score   support

    critical       0.75      0.56      0.64      1347
       major       0.90      0.98      0.94     15349
       minor       0.58      0.04      0.07      1041

    accuracy                           0.89     17737
   macro avg       0.74      0.53      0.55     17737
weighted avg       0.87      0.89      0.87     17737



In [37]:
pd.set_option("display.max_colwidth", 100)

test_view = test.assign(predicted=predictions_balanced)
missed = test_view[(test_view["severity"] == "critical") & (test_view["predicted"] != "critical")]

print("Bugs critiques ratés :", len(missed))
missed[["Short Description", "predicted"]].sample(15, random_state=1)

Bugs critiques ratés : 332


,Short Description,predicted
25249,"Freeze and OOM kill happens in gfxPangoFontGroup::CreateGlyphRunsItemizing, on password input field",major
52452,Freeze when visiting Google Data Center via Google StreetView,major
57564,Phishing protection does not seem to be working properly in Firefox 3; this is a serious regress...,major
4052,Failure to debug-attach on Windows using GDB,major
79188,Manual proxy configuration cannot persist proxy authentication,major
59991,Error: uncaught exception: Permission denied to get property ChromeWindow.Array,major
7004,"When I click ""run the tests"", the browser hangs after the tests run",major
9647,[@ ns4xPluginInstance::Print],major
69129,Internal compiler error caused by a for loop in an initializer block,major
84638,problem with core.resources bundle in I20110124-0916,major


In [38]:
strong = "crash|freeze|hang|oom|use-after-free|assertion|abort|leak"
missed["Short Description"].fillna("").str.contains(strong, case=False).mean()

np.float64(0.14759036144578314)

## Conclusions de l'analyse d'erreurs

Données : 88 682 bugs réels, sévérité regroupée en 3 niveaux (critical, major, minor). Examen sur 17 737 bugs jamais vus par le modèle.

| Robot | Score global | Critical : rappel | Critical : précision |
|---|---|---|---|
| Baseline : répondre « major » à tout | 86,5 % | 0 % | – |
| v1 : règles écrites à la main | 17,9 % | 47 % | – |
| v2 : appris (TF-IDF + régression logistique) | 89,4 % | 56 % | 75 % |
| v2 équilibré | 72,7 % | 75 % | 45 % |

- **Robot retenu** : le v2 équilibré, si rater un bug critique coûte plus cher qu'une fausse alerte. Il trouve 3 bugs critiques sur 4, mais plus de la moitié de ses alertes critiques sont fausses. Le v2 est plus précis (0,75) mais en rate près de la moitié. Le choix dépend du coût des erreurs.
- **Ce que le score global cache** : 86,5 % des bugs sont « major », donc le 89,4 % du v2 ne bat la baseline que de 2,9 points. Il ignore presque les bugs « minor » (rappel de 4 %).
- **Ce que le robot rate** : sur 332 bugs critiques ratés, seuls 15 % contiennent dans leur titre un mot fort (crash, freeze, hang…). Pour les autres, le titre ne porte aucun signal de gravité.
- **Limites des données** : le robot ne lit que le titre ; les classes sont très déséquilibrées ; la sévérité est choisie par la personne qui signale le bug, donc le corrigé contient du bruit (certains classements sont discutables).
- **Pistes d'amélioration** : donner au modèle plus d'informations (description, composant, étapes de reproduction) ; une convention d'écriture des titres, à mesurer par un test avant/après ; regrouper les formes d'un même mot (crash, crashed) ; tester d'autres réglages de l'équilibrage.

In [45]:
eclipse["Project"].value_counts()

Project
Core           46916
Platform       13842
Firefox        11737
JDT             5891
CDT             4228
Thunderbird     3663
Bugzilla        2405
Name: count, dtype: int64

In [46]:
test_view["correct"] = test_view["predicted"] == test_view["severity"]
test_view.groupby("Project")["correct"].agg(["mean", "count"])

,mean,count
Project,,
Bugzilla,0.507400,473
CDT,0.663584,865
Core,0.779210,9389
Firefox,0.706508,2351
JDT,0.622896,1188
Platform,0.697436,2730
Thunderbird,0.630229,741


## Résultat par projet

Le score du v2 équilibré varie de 51 % (Bugzilla, 473 bugs) à 78 % (Core, 9 389 bugs). Le score global de 73 % ne représente pas le comportement sur chaque projet : Core, qui pèse 53 % de l'examen, le tire vers le haut. À vérifier : la répartition des gravités dans chaque projet, qui peut expliquer une partie de l'écart.

In [47]:
df["correct"].mean()

np.float64(0.6666666666666666)